In [1]:
#!/usr/bin/env python
import copy
import coffea
import numpy as np
import awkward as ak
import json
import hist
import yaml

# from mt2 import mt2

from coffea import processor
from coffea.analysis_tools import PackedSelection
from coffea.nanoevents.methods import vector
from coffea.lumi_tools import LumiMask

# silence warnings due to using NanoGEN instead of full NanoAOD
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

from topcoffea.modules.paths import topcoffea_path
from topcoffea.modules.histEFT import HistEFT
import topcoffea.modules.eft_helper as efth
import topcoffea.modules.corrections as tc_cor
import topcoffea.modules.event_selection as tc_es

from ttbarEFT.modules.paths import ttbarEFT_path
from ttbarEFT.modules.analysis_tools import make_mt2, get_lumi
import ttbarEFT.modules.object_selection as tt_os
import ttbarEFT.modules.event_selection as tt_es
import ttbarEFT.modules.corrections as tt_cor 
from ttbarEFT.modules.processor_tools import calc_eft_weights
from ttbarEFT.modules.processor_tools import get_syst_lists

from ttbarEFT.modules import plotting_tools_histEFT as plt_tools

from topcoffea.modules.get_param_from_jsons import GetParam

get_tc_param = GetParam(topcoffea_path("params/params.json"))
get_tt_param = GetParam(ttbarEFT_path("params/params.json"))

NanoAODSchema.warn_missing_crossrefs = False
np.seterr(divide='ignore', invalid='ignore', over='ignore')

import pickle, gzip
import matplotlib.pyplot as plt
from cycler import cycler
import mplhep as hep

import hist
from hist import Hist

from topcoffea.modules.histEFT import HistEFT

In [2]:
def load_pickle(fname):
    return pickle.load(gzip.open(fname))

In [7]:
# hist_dict = load_pickle("SR_ALL_260510.pkl.gz")
hist_dict = load_pickle("SR_ALL_2j3j_260519.pkl.gz")

In [33]:
set(MC_procs) == set(TT_procs+tW_procs+DY_procs+Others_procs)

True

In [4]:
for ch in hist_dict.keys():
    var = 'mllbb'
    hists = hist_dict[ch][var].as_hist({})
    channels = list(hists.axes['channel'])
    all_procs = list(hists.axes['process'])
    MC_procs = [x for x in all_procs if ('data' not in x)]
    
    TT_procs = [x for x in MC_procs if "TT01j2l" in x]
    tW_procs = [x for x in MC_procs if "TW" in x]
    DY_procs = [x for x in MC_procs if "DY" in x]
    TTGJets_procs = [x for x in MC_procs if "TTG" in x]
    ttW_procs = [x for x in MC_procs if "ttW" in x]
    ttZ_procs = [x for x in MC_procs if "ttZ" in x]
    Others_procs = [x for x in MC_procs if ((x not in TT_procs) and (x not in tW_procs) and (x not in DY_procs))]
    
    for chan in channels: 
        print(f"channel: {chan}")        
        TT_vals = hists[{'systematic':'nominal', 'process':TT_procs, 'channel':chan}][{'process':sum}].values()
        tW_Vals = hists[{'systematic':'nominal', 'process':tW_procs, 'channel':chan}][{'process':sum}].values()
        DY_vals = hists[{'systematic':'nominal', 'process':DY_procs, 'channel':chan}][{'process':sum}].values()
        TTG_vals = hists[{'systematic':'nominal', 'process':TTGJets_procs, 'channel':chan}][{'process':sum}].values()
        ttW_vals = hists[{'systematic':'nominal', 'process':ttW_procs, 'channel':chan}][{'process':sum}].values()
        ttZ_vals = hists[{'systematic':'nominal', 'process':ttZ_procs, 'channel':chan}][{'process':sum}].values()
        Others_vals = hists[{'systematic':'nominal', 'process':Others_procs, 'channel':chan}][{'process':sum}].values()

        total = TT_vals[TT_vals >= 0].sum() + tW_Vals[tW_Vals >= 0].sum() + DY_vals[DY_vals >= 0].sum() + Others_vals[Others_vals >= 0].sum()
        print(f"total events: {total}")
        print(f"\t ttbar events: {TT_vals[TT_vals >= 0].sum()}")
        print(f"\t tW events: {tW_Vals[tW_Vals >= 0].sum()}")
        print(f"\t DY events: {DY_vals[DY_vals >= 0].sum()}")
        print(f"\t Others events: {Others_vals[Others_vals >= 0].sum()}")
        print(f"\t TTG events: {TTG_vals[TTG_vals >= 0].sum()}")
        print(f"\t ttW events: {ttW_vals[ttW_vals >= 0].sum()}")
        print(f"\t ttZ events: {ttZ_vals[ttZ_vals >= 0].sum()}")
        # print(f"total events: {ak.sum(nom_hist.values())}")
        # print(f"\t ttbar events: {ak.sum(hists[{'systematic':'nominal', 'process':TT_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t tW events: {ak.sum(hists[{'systematic':'nominal', 'process':tW_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t DY events: {ak.sum(hists[{'systematic':'nominal', 'process':DY_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t Others events: {ak.sum(hists[{'systematic':'nominal', 'process':Others_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t TTG events: {ak.sum(hists[{'systematic':'nominal', 'process':TTGJets_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t TTZ events: {ak.sum(hists[{'systematic':'nominal', 'process':ttZ_procs, 'channel':chan}][{'process':sum}].values())}")
        # print(f"\t TTW events: {ak.sum(hists[{'systematic':'nominal', 'process':ttW_procs, 'channel':chan}][{'process':sum}].values())}")

channel: ee_2b_2j
total events: 12950.337281207925
	 ttbar events: 12320.691325374672
	 tW events: 477.20186153999185
	 DY events: 77.0286810266396
	 Others events: 75.4154132666231
	 TTG events: 58.7308937808263
	 ttW events: 6.792326754871658
	 ttZ events: 7.850550013971909
channel: ee_2b_3j
total events: 14580.171138853622
	 ttbar events: 13882.290940126155
	 tW events: 473.01218645597663
	 DY events: 42.845016124722164
	 Others events: 182.0229961467684
	 TTG events: 124.38594419880654
	 ttW events: 26.735895032701976
	 ttZ events: 26.57557794694608


/users/hnelson2/micromamba/envs/coffea2025/lib/python3.13/site-packages/hist/basehist.py:499: UserWarning: List indexing selection is experimental. Removed bins are not placed in overflow.
  return super().__getitem__(self._index_transform(index))


channel: mm_2b_2j
total events: 10935.925889970533
	 ttbar events: 10355.808376385154
	 tW events: 435.4291572168512
	 DY events: 65.62624711715235
	 Others events: 79.06210925137623
	 TTG events: 60.71205552290335
	 ttW events: 6.494649939974224
	 ttZ events: 8.494873026096785
channel: mm_2b_3j
total events: 13301.900995697973
	 ttbar events: 12585.55814881369
	 tW events: 471.9196486507127
	 DY events: 84.21899203308453
	 Others events: 160.20420620048594
	 TTG events: 100.03970618621499
	 ttW events: 27.261715327927547
	 ttZ events: 28.728143475423217
channel: em_2b_2j
total events: 46886.979280122636
	 ttbar events: 44946.24693040926
	 tW events: 1694.2456262055425
	 DY events: 12.156640009000228
	 Others events: 234.33008349883627
	 TTG events: 183.01063870070945
	 ttW events: 20.319308559552063
	 ttZ events: 21.371444473445344
channel: em_2b_3j
total events: 48914.22536994065
	 ttbar events: 46877.24147027115
	 tW events: 1504.7915868517605
	 DY events: 17.275533711790217
	 Other

In [5]:
def get_shape_syst_arrs(base_histo, syst_var_lst, PDF_var_histo=None):
    # Mapping between systematic name years and process axis year tags
    year_naming = {
        "2016APV": "UL16APV",
        "2016": "UL16",
        "2017": "UL17",
        "2018": "UL18",
    }

    p_arr_rel_lst = []
    m_arr_rel_lst = []

    for syst_name in syst_var_lst:
        if syst_name in ["renormfact", "LHEPDFweight"]:
            continue

        # --- PDF Uncertainty Handling ---
        if syst_name == "PDF" and PDF_var_histo:
            print("running PDF uncertainties")
            relevant_samples_lst = list(PDF_var_histo.axes["process"])
            h = PDF_var_histo[{"process": relevant_samples_lst}]

            nominal = h[{'PDFindex': 0}].values()
            variations = h[{"PDFindex": slice(1, None)}].values()

            diff_sq = np.square(variations - nominal[..., np.newaxis])
            sigma_pdf = np.sqrt(np.sum(diff_sq, axis=-1))

            total_sigma_pdf = np.sum(sigma_pdf, axis=0)

            # PDF uncertainties are symmetric; append directly to quadrature lists
            p_arr_rel_lst.append(total_sigma_pdf**2)
            m_arr_rel_lst.append(total_sigma_pdf**2)
            continue
            
        elif syst_name == "PDF": 
            print("NOT running PDF uncertainties")
            continue

        # --- Standard & Year-Specific Systematic Handling ---
        
        # Determine if this systematic is tied to a specific year
        syst_year_suffix = None
        target_process_year = None
        for yr_syst, yr_proc in year_naming.items():
            if yr_syst in syst_name:
                syst_year_suffix = yr_syst
                target_process_year = yr_proc
                break

        # Identify all relevant samples that have this systematic variation
        h_up = base_histo[{"systematic": syst_name + "Up"}]
        all_relevant_samples = list(h_up.axes["process"])

        if syst_year_suffix:
            # Target exactly what should vary
            if target_process_year == "UL16":
                # For 2016, look for UL16 but explicitly ignore UL16APV
                var_samples = [s for s in all_relevant_samples if "UL16" in s and "UL16APV" not in s]
            else:
                # For 2016APV, 2017, 2018, standard substring match works flawlessly
                var_samples = [s for s in all_relevant_samples if target_process_year in s]
            
            # Anything that isn't varying MUST be kept as nominal.
            nom_samples = [s for s in all_relevant_samples if s not in var_samples]

            # Total Nominal (all relevant samples across all years)
            n_arr = base_histo[{"process": all_relevant_samples, "systematic": "nominal"}]
            n_arr = n_arr[{"process": sum}].values()

            # Calculate Up Variation: (Target Year Up) + (Other Years Nominal)
            u_arr_var_year = base_histo[{"process": var_samples, "systematic": syst_name + "Up"}][{"process": sum}].values()
            u_arr_nom_years = base_histo[{"process": nom_samples, "systematic": "nominal"}][{"process": sum}].values() if nom_samples else 0
            u_arr_sum = u_arr_var_year + u_arr_nom_years

            # Calculate Down Variation: (Target Year Down) + (Other Years Nominal)
            d_arr_var_year = base_histo[{"process": var_samples, "systematic": syst_name + "Down"}][{"process": sum}].values()
            d_arr_nom_years = base_histo[{"process": nom_samples, "systematic": "nominal"}][{"process": sum}].values() if nom_samples else 0
            d_arr_sum = d_arr_var_year + d_arr_nom_years

        else:
            # Standard treatment for year-correlated
            n_arr = base_histo[{"process": all_relevant_samples, "systematic": "nominal"}][{"process": sum}].values()
            u_arr_sum = base_histo[{"process": all_relevant_samples, "systematic": syst_name + "Up"}][{"process": sum}].values()
            d_arr_sum = base_histo[{"process": all_relevant_samples, "systematic": syst_name + "Down"}][{"process": sum}].values()

        # Differences relative to nominal
        u_arr_rel = u_arr_sum - n_arr
        d_arr_rel = d_arr_sum - n_arr
        
        # Envelope: Extract strictly positive and negative shifts relative to 0
        p_arr_rel = np.maximum.reduce([u_arr_rel, d_arr_rel, np.zeros_like(n_arr)])
        m_arr_rel = np.minimum.reduce([u_arr_rel, d_arr_rel, np.zeros_like(n_arr)])
        
        # Add in quadrature
        p_arr_rel_lst.append(p_arr_rel**2)
        m_arr_rel_lst.append(m_arr_rel**2)

    return [np.sum(m_arr_rel_lst, axis=0), np.sum(p_arr_rel_lst, axis=0)]

def get_shape_syst_lst(base_histo, ignore_list=[]):
    # Get unique systematic base names (e.g., "ISR" from "ISRUp")
    all_syst_var_lst = list(base_histo.axes["systematic"])
    syst_var_lst = []
    for name in all_syst_var_lst:
        if name.endswith("Up"):
            base_name = name[:-2] # strip "Up" to get the base systematic name
            if base_name in ignore_list: continue
            if base_name not in syst_var_lst:
                syst_var_lst.append(base_name)
                
    return syst_var_lst

year_naming = {
    "2016APV": "UL16APV",
    "2016": "UL16",
    "2017": "UL17",
    "2018": "UL18",
}

In [9]:
with open("/users/hnelson2/ttbarEFT-coffea2025/ttbarEFT/params/ttLOuncert.json", "r") as f:
    powheg_unc_data = json.load(f)

var = 'mllbb'
for d in hist_dict.keys():
    base_hist = hist_dict[d][var]
    MC_procs = [x for x in list(base_hist.axes['process']) if 'data' not in x]
    data_procs = [x for x in list(base_hist.axes['process']) if 'data' in x]
    channels = list(base_hist.axes['channel'])

    for ch in channels: 
        MC_hist = base_hist[{'systematic':'nominal', 'channel':ch, 'process':MC_procs}][{'process':sum}].as_hist({})
        mc_err = base_hist[{'systematic':'sumw2', 'channel':ch, 'process':MC_procs}][{'process': sum}].as_hist({}).values()

        syst_to_apply = get_shape_syst_lst(base_hist[{'channel':ch, 'process':MC_procs}].as_hist({}))
        h_PDFweights = hist_dict[d]['LHEPDFweights'][{'channel':ch}]
        shape_systs_summed_arr_m , shape_systs_summed_arr_p = get_shape_syst_arrs(base_hist[{'channel':ch, 'process':MC_procs}].as_hist({}), syst_var_lst=syst_to_apply, PDF_var_histo=h_PDFweights)

        nom_arr_all = MC_hist.values() 
        json_variance_p = np.zeros_like(nom_arr_all)
        json_variance_m = np.zeros_like(nom_arr_all)
        
        if var in powheg_unc_data and ch in powheg_unc_data[var]:
            print(f"including powheg_unc_data")
            ch_data = powheg_unc_data[var][ch]
            json_nom  = np.array(ch_data["nominal"][1:-1])
            json_up   = np.array(ch_data["up"][1:-1])
            json_down = np.array(ch_data["down"][1:-1])
            
            delta_p = json_up - json_nom
            delta_m = json_nom - json_down
            
            json_variance_p = delta_p ** 2
            json_variance_m = delta_m ** 2
        
        global_yield = np.sum(nom_arr_all)
        global_uperr = np.sqrt(np.sum(shape_systs_summed_arr_p + mc_err + json_variance_p))
        global_downerr = np.sqrt(np.sum(shape_systs_summed_arr_m + mc_err + json_variance_m))
        
        print(f"channel: {ch}")
        print(f"\t yield: {global_yield}")
        print(f"\t sigma up: {global_uperr}")
        print(f"\t sigma do: {global_downerr} \n\n")

including powheg_unc_data
channel: ee_2b_2j
	 yield: 12940.279681440026
	 sigma up: 267.75624710750276
	 sigma do: 280.2885136665655 


including powheg_unc_data
channel: ee_2b_3j
	 yield: 14570.467890125665
	 sigma up: 395.1624954345435
	 sigma do: 421.2429755702144 


including powheg_unc_data
channel: mm_2b_2j
	 yield: 10925.230141394828
	 sigma up: 221.52215206863082
	 sigma do: 220.1888118771857 


including powheg_unc_data
channel: mm_2b_3j
	 yield: 13297.3091520579
	 sigma up: 353.67668370832047
	 sigma do: 370.27953923574177 


including powheg_unc_data
channel: em_2b_2j
	 yield: 46883.68765576217
	 sigma up: 916.8475728705579
	 sigma do: 904.4458836840507 


including powheg_unc_data
channel: em_2b_3j
	 yield: 48910.541064261306
	 sigma up: 1227.805285750602
	 sigma do: 1322.5084998879843 


